# Explore the fixture data

A step-by-step look at the synthetic sales data the project will run on: what the
series look like, what the forecasts look like, how they are scored, and how the
planted faults show up.

The data comes from `06_generator`, so this is exactly what the agent will work on.
With `SEED = 7` it is the snapshot saved in `cases/dev/fam_01/`; change `SEED` and
re-run to get a different draw.

Run the cells top to bottom (Shift+Enter). Nothing here is exported.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from agentic_forecast_review.generator import (
    CATEGORY_LEVEL,
    ISSUE_DAYS,
    LAST_DAY,
    generate,
)
from agentic_forecast_review.metrics import mae, signed_bias

SEED = 7
PERIOD = (169, 196)  # review period: is the forecast OK here?
WINDOW = (141, 168)  # comparison window: the 4 weeks before
pd.set_option("display.width", 120)

## 2. The actual sales

Made by `06_generator`: each series = store level × category level × weekly pattern ×
slow trend + noise, rounded to whole units and never below 0. Every row has a `row_id`
that a finding can cite.

In [ ]:
snap = generate(SEED)
actuals = pd.DataFrame(snap.actuals)
print(snap.snapshot_id)
actuals.head(10)

How big is it? One row per series per day.

In [ ]:
print("rows:", len(actuals))
print("series:", actuals["series_id"].nunique())
print("days:", actuals["day"].min(), "to", actuals["day"].max())
assert len(actuals) == 30 * LAST_DAY

## 3. Summary per series

Average, spread and range of daily sales. FOODS sells most, HOBBIES least; bigger
stores sell more of everything.

In [ ]:
summary = (
    actuals.groupby(["category", "series_id"])["value"]
    .agg(["mean", "std", "min", "max"])
    .round(1)
)
summary

The same by category only.

In [ ]:
actuals.groupby("category")["value"].describe().round(1)

## 4. One store over time

Pick any series ID from the table above. Dashed lines are the forecast issue days;
shading marks the comparison window (grey) and the review period (yellow).

In [ ]:
STORE = "CA_2"

fig, ax = plt.subplots(figsize=(11, 4))
for category in CATEGORY_LEVEL:
    one = actuals[actuals["series_id"] == f"{STORE}_{category}"]
    ax.plot(one["day"], one["value"], lw=1, label=f"{STORE}_{category}")
for day in ISSUE_DAYS:
    ax.axvline(day, color="grey", ls="--", lw=0.8)
ax.axvspan(*WINDOW, color="grey", alpha=0.12, label="comparison window")
ax.axvspan(*PERIOD, color="gold", alpha=0.2, label="review period")
ax.set(xlabel="day", ylabel="units sold", title=f"Daily sales, {STORE}")
ax.legend(fontsize=8, ncol=5)
plt.show()

## 5. The weekly pattern

Average sales by weekday, as a share of each series' own average. Weekends (Sat, Sun)
should sit about 20–25% above the mid-week dip.

In [ ]:
weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
actuals["weekday"] = [weekday_names[(d - 1) % 7] for d in actuals["day"]]
actuals["relative"] = actuals["value"] / actuals.groupby("series_id")[
    "value"
].transform("mean")
weekly = (
    actuals.pivot_table(index="weekday", columns="category", values="relative")
    .reindex(weekday_names)
    .round(2)
)
weekly

In [ ]:
weekly.plot(
    kind="bar", rot=0, figsize=(8, 3.5), title="Sales by weekday (1.0 = series average)"
)
plt.axhline(1.0, color="black", lw=0.6)
plt.show()

## 6. The trend

Weekly totals smooth out the weekday pattern so the slow drift is visible.

In [ ]:
actuals["week"] = (actuals["day"] - 1) // 7 + 1
weekly_totals = actuals.pivot_table(
    index="week", columns="category", values="value", aggfunc="sum"
)
weekly_totals.plot(figsize=(9, 3.5), title="Weekly units sold, all stores")
plt.show()

## 7. The forecasts

Two models, each issued on days 112, 140 and 168 for the next 28 days.

- `seasonal_naive_v1`: repeat the last observed week.
- `synthetic_v3`: the same plus a small drift and noise, so the two models differ a little.

In [ ]:
values = {
    series_id: group.set_index("day")["value"]
    for series_id, group in actuals.groupby("series_id")
}
forecasts = pd.DataFrame(snap.forecasts)
print("forecast rows:", len(forecasts))
forecasts.head(10)

## 8. Forecast vs actual in the review period

The forecast issued on day 168 covers exactly the review period 169–196.

In [ ]:
SERIES = "CA_2_FOODS"


def review_frame(fc: pd.DataFrame, series_id: str, issue_day: int) -> pd.DataFrame:
    """Actual and both forecasts side by side for one series and issue day."""
    picked = fc[(fc["series_id"] == series_id) & (fc["issue_day"] == issue_day)]
    wide = picked.pivot_table(
        index="target_day", columns="model_version", values="value"
    )
    wide["actual"] = values[series_id].reindex(wide.index)
    return wide


period_view = review_frame(forecasts, SERIES, 168)
period_view.head(7)

In [ ]:
ax = period_view.plot(figsize=(10, 3.8), style={"actual": "k-o"}, ms=3)
ax.set(xlabel="day", ylabel="units sold", title=f"{SERIES}: forecasts issued day 168")
ax.legend(loc="upper left", fontsize=8)
plt.show()

## 9. Scoring: MAE and bias

Using the project's own `mae` and `signed_bias` (from `01_metrics`): for every series
and model, score the review period (forecast from day 168) and the comparison window
(forecast from day 140). A big jump from window to period is what a review would chase.

In [ ]:
def score(fc: pd.DataFrame, issue_day: int) -> pd.DataFrame:
    out = []
    for (series_id, model), group in fc[fc["issue_day"] == issue_day].groupby(
        ["series_id", "model_version"]
    ):
        actual = values[series_id].reindex(group["target_day"]).tolist()
        predicted = group["value"].tolist()
        out.append(
            {
                "series_id": series_id,
                "model_version": model,
                "mae": round(mae(actual, predicted), 2),
                "bias": round(signed_bias(actual, predicted), 2),
            }
        )
    return pd.DataFrame(out).set_index(["series_id", "model_version"])


scores = pd.merge(
    score(forecasts, 168),
    score(forecasts, 140),
    left_index=True,
    right_index=True,
    suffixes=("_period", "_window"),
)
scores["mae_delta"] = (scores["mae_period"] - scores["mae_window"]).round(2)
scores.sort_values("mae_delta", ascending=False).head(10)

Bias near 0 means no systematic over- or under-forecasting. On clean data it stays
small (a unit or two either way) for every series.

In [ ]:
scores["bias_period"].unstack("model_version").plot(
    kind="hist", bins=20, alpha=0.6, figsize=(8, 3), title="Bias in the review period"
)
plt.show()

## 10. What the planted faults look like

The generator will damage one family's data in exactly one way. Two examples here,
applied to copies so the clean data above is untouched.

**missing_dates**: delete 4 actual days for one series in the review period. The
comparison then has 24 actuals instead of 28; `check_quality` should name the gap.

In [ ]:
missing_days = [175, 176, 182, 189]
damaged = actuals[
    ~((actuals["series_id"] == SERIES) & actuals["day"].isin(missing_days))
]
in_period = damaged[(damaged["series_id"] == SERIES) & damaged["day"].between(*PERIOD)]
print("actual days present in period:", len(in_period), "of", PERIOD[1] - PERIOD[0] + 1)
print("missing:", sorted(set(range(PERIOD[0], PERIOD[1] + 1)) - set(in_period["day"])))

**bias_shift**: multiply `synthetic_v3` by 0.85 for one category in the period. Bias
turns clearly negative (forecast too low) for every series in that category.

In [ ]:
shifted = forecasts.copy()
mask = (
    (shifted["model_version"] == "synthetic_v3")
    & (shifted["issue_day"] == 168)
    & shifted["series_id"].str.endswith("_FOODS")
)
shifted.loc[mask, "value"] = (shifted.loc[mask, "value"] * 0.85).round()
compare = pd.DataFrame(
    {
        "clean": score(forecasts, 168)["bias"],
        "bias_shift": score(shifted, 168)["bias"],
    }
).xs("synthetic_v3", level="model_version")
compare["category"] = compare.index.str.split("_").str[-1]
compare.groupby("category")[["clean", "bias_shift"]].mean().round(2)

## 11. Things to try

- Change `SEED` in step 1 and re-run everything: the shape stays, the numbers move.
- Change `STORE` (step 4) or `SERIES` (step 8) to look at another series.
- In step 10, change `0.85` to `0.95`: how small a shift still stands out?